# Movie Recommendation System
### TMDB 5000 Dataset — Content-Based Filtering

The idea here is simple: given a movie, find the 5 most similar ones based on its genres, keywords, cast, director, and plot. No ratings or user data involved — just the movie's own metadata.

The approach: pull all that info together into one text "tag" per movie, vectorise it, then use cosine similarity to find the closest matches.


## 1. Imports

In [1]:
import pandas as pd
import numpy as np

## 2. Loading the data

Two files — one with general movie info (genres, budget, overview etc.) and one with cast and crew. We need both.


In [2]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [3]:
credits.head(2)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


## 3. Merging

Joining them on `title` so everything is in one place. `movie_id` from the credits file becomes the unique ID we use going forward.


In [4]:
movies = movies.merge(credits, on = 'title') #merge both columns on the basis of title, it won't get added twice
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [5]:
movies.shape

(4809, 23)

In [6]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

## 4. Cutting it down to what we actually need

Most of those 23 columns aren't useful here. We only care about genres, keywords, the plot overview, cast, crew (just the director), title, and movie_id.


In [7]:
#final dataframe after filtering variables
movies = movies[['genres', 'movie_id', 'keywords', 'overview', 'title', 'cast', 'crew']]


In [8]:
movies.isnull().sum()

genres      0
movie_id    0
keywords    0
overview    3
title       0
cast        0
crew        0
dtype: int64

## 5. Dropping nulls and duplicates

Quick data quality check before we start doing anything with it.

In [9]:
movies.dropna(inplace=True)

In [10]:
movies.duplicated().sum()

np.int64(0)

## 6. Parsing the JSON columns

`genres`, `keywords`, `cast`, and `crew` are all stored as JSON strings — lists of dicts like `[{"id": 28, "name": "Action"}, ...]`. We need to pull out just the names.


In [11]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

The `convert()` function handles this — just iterates through and grabs the `name` field from each dict.

In [12]:
import ast
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [13]:
movies['genres'] = movies['genres'].apply(convert)

In [14]:
movies['keywords'] = movies['keywords'].apply(convert)

In [15]:
movies.head(2)

,genres,movie_id,keywords,overview,title,cast,crew
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...",Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug abuse, exotic island, east india ...","Captain Barbossa, long believed to be dead, ha...",Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


For cast we only take the top 3 actors. The full cast list would add a lot of noise,  minor roles aren't going to help match similar movies.

In [16]:
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter <=2:
            L.append(i['name'])
            counter +=1
        else:
            break
    return L 

In [17]:
movies['cast'] = movies['cast'].apply(convert3)

For crew, we only want the director. There are dozens of roles in there (producer, editor, etc.) but the director is the one that really defines the film's style.

In [18]:
def fetch_director(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if i['job'] == "Director":
            L.append(i['name'])
            break
    return L

In [19]:
movies['crew'] = movies['crew'].apply(fetch_director)

## 7. Tidying up the text

First, split the overview string into a word list so we can concatenate it with the other columns later.


In [20]:
#we want to change 'overview' column from string to list so that we can concatenate it to other list
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [21]:
movies.head(2)

,genres,movie_id,keywords,overview,title,cast,crew
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...","[In, the, 22nd, century,, a, paraplegic, Marin...",Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug abuse, exotic island, east india ...","[Captain, Barbossa,, long, believed, to, be, d...",Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]


Then we remove spaces from multi-word entries  so `"Zoe Saldana"` becomes `"ZoeSaldana"` and `"Science Fiction"` becomes `"ScienceFiction"`. 

The reason: if we leave the spaces in, the vectoriser splits them into individual words. Then "Zoe" from one movie might match "Zoe" from a completely unrelated one.


In [22]:
movies['genres'] = movies['genres'].apply(lambda x : [i.replace( " ", "") for i in x])

In [23]:
movies['keywords'] = movies['keywords'].apply(lambda x : [i.replace( " ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x : [i.replace( " ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x : [i.replace( " ", "") for i in x])

In [24]:
movies.head()

,genres,movie_id,keywords,overview,title,cast,crew
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...","[In, the, 22nd, century,, a, paraplegic, Marin...",Avatar,"[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,"[Adventure, Fantasy, Action]",285,"[ocean, drugabuse, exoticisland, eastindiatrad...","[Captain, Barbossa,, long, believed, to, be, d...",Pirates of the Caribbean: At World's End,"[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,"[Action, Adventure, Crime]",206647,"[spy, basedonnovel, secretagent, sequel, mi6, ...","[A, cryptic, message, from, Bond’s, past, send...",Spectre,"[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,"[Action, Crime, Drama, Thriller]",49026,"[dccomics, crimefighter, terrorist, secretiden...","[Following, the, death, of, District, Attorney...",The Dark Knight Rises,"[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,"[Action, Adventure, ScienceFiction]",49529,"[basedonnovel, mars, medallion, spacetravel, p...","[John, Carter, is, a, war-weary,, former, mili...",John Carter,"[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


## 8. Building the tags

Now we just concatenate overview + genres + director + keywords + cast  into one big list per movie, then join it into a lowercase string.

This single `tags` field is what the whole similarity model is based on.


In [25]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['crew'] + movies['keywords'] + movies['cast']

In [26]:
new_df = movies[['movie_id', 'title', 'tags']]

In [27]:
#convert tags to string
new_df['tags'] = new_df['tags'].apply(lambda x : " ".join(x))

C:\Users\HP\AppData\Local\Temp\ipykernel_19564\2726124483.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x : " ".join(x))


In [28]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

C:\Users\HP\AppData\Local\Temp\ipykernel_19564\3214958533.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


In [29]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


## 9. Stemming

Words like "dancing", "danced", and "dance" mean the same thing but the vectoriser would count them separately. Stemming reduces each word to its root so they all collapse to the same token.

Using NLTK's Porter Stemmer for this.


In [30]:
!pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [32]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [33]:
stem('dancing danced dance')

'danc danc danc'

In [34]:
new_df['tags'] = new_df['tags'].apply(stem)

C:\Users\HP\AppData\Local\Temp\ipykernel_19564\3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [35]:
new_df['tags'][133]

'vampir barnaba collin is inadvert freed from hi tomb and emerg into the veri chang world of 1972. he return to collinwood manor to find that hi once-grand estat and famili have fallen into ruin. comedi fantasi timburton witch imprison vampir curs fishoutofwat chain gothic mad oldhous lostlov angrymob 18thcenturi ghost hiddenroom oldmans johnnydepp michellepfeiff helenabonhamcart'

## 10. Vectorising

Converting the tags into numbers using CountVectorizer - basically just counting how often each word appears per movie. We cap it at the top 5000 words and strip out English stop words (the, a, is, etc.).

Each movie ends up as a vector with 5000 dimensions.


In [36]:
#repeat the steps from 41st cell to create new vectors
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 5000, stop_words='english')

In [37]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [38]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(5000,))

In [39]:
cv.get_feature_names_out()[:100]

array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '18', '18th', '19', '1910', '1920', '1930', '1940', '1950',
       '1950s', '1960', '1960s', '1970', '1970s', '1974', '1976', '1980',
       '1985', '1990', '1999', '19th', '19thcenturi', '20', '200', '2003',
       '2009', '20th', '21st', '23', '24', '25', '30', '300', '3d', '40',
       '50', '500', '60', '70', '80', 'aaron', 'aaroneckhart', 'abandon',
       'abduct', 'abigailbreslin', 'abil', 'abl', 'aboard', 'abov',
       'abus', 'academ', 'academi', 'accept', 'access', 'accid',
       'accident', 'acclaim', 'accompani', 'accomplish', 'account',
       'accus', 'ace', 'achiev', 'acquaint', 'act', 'action',
       'actionhero', 'activ', 'activist', 'activities', 'actor',
       'actress', 'actual', 'ad', 'adam', 'adamsandl', 'adamshankman',
       'adapt', 'add', 'addict', 'adjust', 'admir', 'admit', 'adolesc',
       'adopt', 'ador', 'adrienbrodi', 'adult', 'adultanim', 'adulteri'],
      dtype=obj

## 11. Cosine similarity

We have 4806 movies, each as a 5000-dim vector. Now we calculate how similar every movie is to every other movie — that's a 4806×4806 matrix.

We use cosine similarity rather than Euclidean distance because in high-dimensional spaces Euclidean distance stops being meaningful — everything ends up roughly the same distance apart. Cosine similarity looks at the angle between vectors instead, which works much better here. A score of 1 means identical, 0 means nothing in common.


In [40]:
from sklearn.metrics.pairwise import cosine_similarity
#if it is 0, similarity is low. if it is 1, highly similar
#passing all vectors into cosine function
similarity = cosine_similarity(vectors)
#shape is giving the number that, for example, take the first movie and 
#compare its similarity with 4805 other movies and same for all the others as well

In [41]:
similarity.shape

(4806, 4806)

## 12. The recommend function

Take a movie name, find its index, look up its row in the similarity matrix, sort by score (descending), skip the first result (which is always the movie itself with a score of 1), return the top 5.


In [42]:
def recommend(movie):
    #fetch the movie index
    movie_index = new_df[new_df['title'] == movie].index[0]
    #fetch distance between then
    distances = similarity[movie_index]
    #sort the list but it will lose its original index position, we will have
    #to not let that happen with enumerate function, it will hold the index position
    #ig. then create a list of them. then pass it on sorted with reverse = true
    #as we need in desc. but it will do desc sorting on the basis of index, not
    #what we need. we do key = lambda ... for that. then narrow it to 6 movies. change
    #similarity[0] to distances as it is already made in the previous line of code
    movie_list = sorted(list(enumerate(distances)), reverse = True, key = lambda x:x[1])[1:6]
    for i in movie_list:
        print(new_df.iloc[i[0]].title)


## 13. Testing it out

In [44]:
recommend('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.
